# Summary Agent Test (Stages 6 + 8)

Full pipeline:
1. Load ChromaDB index (retriever)
2. Retrieve relevant chunks for a question
3. Generate **executive summary** and **key insights** via `SummaryAgent`

**Prerequisites:**
- Index already created (`01_test_rag_pipeline.ipynb`, Part A)
- `.env` with `HUGGINGFACE_API_TOKEN` (Stage 7)
- `uv sync`

## Configuration

In [8]:
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VECTOR_DB_PATH = ROOT / "data/vector_db"

TOP_K = 4
QUESTION = "What is the main contribution of this paper?"

token = os.getenv("HUGGINGFACE_API_TOKEN") or os.getenv("HF_TOKEN")

print(f"Project root: {ROOT}")
print(f"Vector DB: {VECTOR_DB_PATH}")
print(f"Default question: {QUESTION}")
print(f"HF token: {'configured' if token else 'NOT configured'}")

Project root: d:\Github\research-workflow-agent
Vector DB: d:\Github\research-workflow-agent\data\vector_db
Default question: What is the main contribution of this paper?
HF token: configured


## 1. Load retriever

In [9]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()

chunk_count = store.collection.count()
print(f"Collection: {store.collection_name} ({chunk_count} chunks)")

if chunk_count == 0:
    raise RuntimeError(
        "Empty index. Run notebook 01_test_rag_pipeline (Part A) first."
    )

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

Collection: book_research (52 chunks)


## 2. Retrieve context (RAG)

In [10]:
context = retriever.invoke(QUESTION)

print(f"Question: {QUESTION}")
print(f"Chunks retrieved: {len(context)}")
print()

for i, doc in enumerate(context, start=1):
    print(f"--- Chunk {i} | page={doc.metadata.get('page')} | distance={doc.metadata.get('distance')} ---")
    print(doc.page_content[:300].replace("\n", " "))
    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2483.36it/s]


Question: What is the main contribution of this paper?
Chunks retrieved: 4

--- Chunk 1 | page=11 | distance=0.42611008882522583 ---
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993. [26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In Proceedings of the 

--- Chunk 2 | page=9 | distance=0.4303792715072632 ---
comments, corrections and inspiration. References [1] Jimmy Lei Ba, Jamie Ryan Kiros, and Geoffrey E Hinton. Layer normalization. arXiv preprint arXiv:1607.06450, 2016. [2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly learning to align and translate. CoRR

--- Chunk 3 | page=8 | distance=0.43116796016693115 ---
Penn Treebank [25], about 40K training sentences. We also trained it in a semi-supervised setting, using the larger high-confidence and BerkleyParser co

## 3. Run Summary Agent

In [11]:
from agents.summary_agent import SummaryAgent

print("Calling Summary Agent (Hugging Face API)...")
agent = SummaryAgent()
result = agent.run(QUESTION, context)

Calling Summary Agent (Hugging Face API)...


In [12]:
print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print(result.executive_summary)
print()
print("=" * 60)
print("KEY INSIGHTS")
print("=" * 60)
for insight in result.key_insights:
    print(f"- {insight}")

EXECUTIVE SUMMARY
The main contribution of this paper is the introduction of a new simple network architecture called the Transformer, which is based solely on attention mechanisms and dispenses with recurrence and convolutions. This architecture is proposed as an alternative to complex recurrent or convolutional neural networks that are commonly used in sequence transduction models.

The Transformer architecture is designed to be more efficient and effective than existing models, and it achieves state-of-the-art results in machine translation tasks. The authors of the paper propose a new way of processing input sequences using self-attention mechanisms, which allows the model to weigh the importance of different input elements and focus on the most relevant ones.

The paper presents a comprehensive evaluation of the Transformer architecture on several machine translation tasks, and it shows that it outperforms existing models in terms of accuracy and efficiency.

KEY INSIGHTS
- The Tr

## 4. Test another question

Change `another_question` and run the cells below.

In [13]:
another_question = "How does self-attention work in the Transformer?"

context2 = retriever.invoke(another_question)
result2 = agent.run(another_question, context2)

print(f"Question: {another_question}")
print()
print("EXECUTIVE SUMMARY:")
print(result2.executive_summary)
print()
print("KEY INSIGHTS:")
for insight in result2.key_insights:
    print(f"- {insight}")

Question: How does self-attention work in the Transformer?

EXECUTIVE SUMMARY:
Self-attention in the Transformer is a mechanism that allows each position in a sequence to attend to all positions in the input or previous layers. This is achieved by using the same keys, values, and queries from the same place, typically the output of the previous layer. In the encoder, self-attention layers enable each position to attend to all positions in the previous layer, while in the decoder, self-attention layers allow each position to attend to all positions in the decoder up to and including that position. This is done to prevent leftward information flow and preserve the auto-regressive property.

The Transformer uses multi-head attention to compute representations of its input and output without using sequence-aligned RNNs or convolution. Self-attention has been successfully used in various tasks, including reading comprehension, abstractive summarization, and textual entailment. The Transform